In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from openai import OpenAI

# Constants
SPREADSHEET_ID = '1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ'  
SHEET_NAME = 'Sheet2'
CREDENTIALS_FILE = 'data/url-to-email-445616-cebe4868914f.json'  
OPENAI_API_KEY = ""
# Category ranking
category_ranking = {
	"TESTIMONIALS": 1,
	"COURSES": 2,
	"SERVICES": 3,
	"WEBINAR": 4,
	"PODCAST": 5,
	"EBOOK": 6,
	"RECENT_BLOG": 7,
	"ABOUT_US": 8,
	"SHOP": 9
}

# Column indices for categories (0-based, A=0, B=1, ..., M=12, N=13, ..., U=20)
category_indices = {
	"ABOUT_US": 12,  # M
	"EBOOK": 13,     # N
	"COURSES": 14,   # O
	"RECENT_BLOG": 15, # P
	"TESTIMONIALS": 16, # Q
	"WEBINAR": 17,   # R
	"SERVICES": 18,  # S
	"PODCAST": 19,   # T
	"SHOP": 20       # U (excluded)
}

# Output column letters
output_columns = {
	"Email 1": "W",
	"Email 1 Data Point": "X",
	"Subsequence 1": "Y",
	"Subsequence 1 Data Point": "Z",
	"Subsequence 2": "AA",
	"Subsequence 2 Data Point": "AB",
	"Subsequence 3": "AC",
	"Subsequence 3 Data Point": "AD",
	"Subsequence 4": "AE",
	"Subsequence 4 Data Point": "AF",
	"Email 2": "AG",
	"Email 3": "AH"
}

# Prompt templates
prompt_templates = {
"Email 1": {
	"TESTIMONIALS": """
		[FIRST NAME] - Saw your work with [PERSONALISATION - SOME INSIGHT FROM ANY OF THE TESTIMONIALS OR REVIEWS] — love those results!

		Looks like you can help a ton more folks make the same leap.

		Assuming, if we could 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your existing stuff and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your paid programs.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"COURSES": """
    [First Name] - Just looked in your [PERSONALISATION - Name of the course or the program, suffix with the word course or program as per the data point and if it feels necessary].  

		Sooo many people could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to ______  real fast.

		Would you like to know more?
	""",
  "SERVICES": """
    [Fist Name] - Just checked out your [PERSONALISATION - Some insight from their data point] page . 

		Sooo many [PERSONALISATION - Think what could be the ICP for this service and add that ICP here in just keywords] could use that!

		Assuming, If we could take the insights & frameworks from it and feature them on our network of High Traffic Pages.

		One client saw a 287% Increase in online course purchases. 

		There’s so much value here, people would go from strangers to flocking to [PERSONALISATION - what type of deals would be possible for the service they are offering for example in real estate it could be multifamily / REIT / Syndicate, think according to the data point what could be the ideal type of deal] real fast.

		Would you like to know more?
	""",
  "WEBINAR": """
   	[FIRST NAME] - Just came across your [PERSONALISATION - Name/topic of the webinar] webinar — the focus on [PERSONALISATION - Some insight which would fit in perfectly with this sentence and tone] sounds like a game-changer!

		This kind of value deserves a much bigger audience.
        
		Assuming we could 10X your exposure in 45 days, without you lifting a finger…

    We’ll do this by using the insights from your [PERSONALISATION - Name/topic of the webinar] webinar and featuring them on our network of High Traffic Pages.
		
		It can build authority and turn strangers into buyers fast, for your paid programs.
		
		One client saw a 287% Increase in online course purchases.
		
		Would you like to know how?
	""",
  "PODCAST": """
    [FIRST NAME] - Just came across your podcast about [PERSONALISATION - some insight from the podcast which really stood out and fits here in the sentence] – honestly, so good!

		The pod deserves a much bigger audience for so much value. 

		What if we could 10X your exposure in 45 days, no effort on your part?

		We’d pull insights from the pod & feature them on our high-traffic network.

		This builds authority and turns strangers into buyers for your paid programs fast. One client saw a 287% Spike in course sales.

		Want to hear how it works?
	""",
	"EBOOK": """
    [FIRST NAME] - Saw your Book on [PERSONALISATION - What is the ebook about, deduce from the title or the description] — I think it’s a great way to get people into your higher ticket stuff.

		Would be great if more people saw it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your book and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"RECENT_BLOG": """
    [FIRST NAME] - Saw your blog on [PERSONALISATION - What is the blog about] — really hits the mark.

		Would be great if more people read it.

		If we could use it to 10X your exposure in 45 days, without you lifting a finger…

		We’ll do this by using the insights from your blog and feature them on our network of High Traffic Pages.

		It can build authority and turn strangers into buyers fast, for your higher ticket programs.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know how?
	""",
	"ABOUT_US": """
    [FIRST NAME] - I looked into [PERSONALISATION - Company Name] and love your [PERSONALISATION - What they do and why we like that - insightful].

		Feels like, a ton of people could use that.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.


		I think there’s so much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases. 

		Would you like to know more?
	""",
	"NEUTRAL": """
    [FIRST NAME] – being someone with your own course/consulting offer.

		I can see how more visibility can help sell more of those.

		Assuming, we could take your existing stuff & 5X your existing exposure in 45 days…

		Without you doing any heavy lifting.

		So much value here, the right people would straight-up flock to it.

		One client saw a 287% Increase in online course purchases / consulting clients. 

		Would you like to know more?
	"""
},
"Subsequence 1": {
	"TESTIMONIALS": """
    Thanks for reaching out! The work you did with [PERSONALISATION - Jason and how you helped him 10x his revenue] actually made me think…

		Can you help more folks make the same leap? Pretty sure more people would want that!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"COURSES": """
    Thanks for reaching out! Your [PERSONALISATION - Name of the course or the program, suffix with the word course or program as per the data point and if it feels necessary] actually made me think…

		Can we use some insights from it too? Folks would be all over it!

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"SERVICES": """
    Thanks for reaching out! Your [PERSONALISATION - some insight that made us think should fit in with the rest of the sentence here] page actually made me think…

		Mind if we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"WEBINAR": """
		Thanks for reaching out! Your [PERSONALISATION - Name/topic of the webinar, remove the word webinar if it is there at the end of the name] webinar actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"PODCAST": """
    Thanks for reaching out! Your podcast about [PERSONALISATION - some insight from the podcast which really stood out and fits here in the sentence] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"EBOOK": """
    Thanks for reaching out! Your Book on [PERSONALISATION - What is the ebook about] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"RECENT_BLOG": """
    Thanks for reaching out! Your blog on [PERSONALISATION - What is the blog about] actually made me think…

		Can we use some insights from it too? Folks would be all over it lol

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"ABOUT_US": """
    Thanks for reaching out! I looked into [COMPANY NAME] and loved your approach [PERSONALISATION - What they do and why we like that - insightful].

		Feels like a ton of people could use that.

		Here’s a video I recorded going into detail on this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	""",
	"NEUTRAL": """
    Appreciate you reaching back out! I think there’s sooo much potential if we do this together.

		Here’s a quick 2 min pre-recorded video (to save myself some time LOL) we recorded going into this: 

		Loom.com/video

		I’d love to find out more about [COMPANY NAME] and how this could 5-10x your exposure. 

		Grab a time with me here: 

		https://calendly.com/scale-brands-lab/30min
	"""
},
"Subsequence 2": """
	I was looking at your [Website] and couldn’t help but notice [PERSONALISATION - Some insight related to how they are helping their ICP]. 
    
	Got some time to talk about it tomorrow?
""",
"Subsequence 3": """
	Knowing that AI’s blowing up in [Industry], [First Name].
    
	Like those crazy chatbots!
    
	I'm curious, what’s your next move to stay ahead of the curve?
""",
"Subsequence 4": """
	I keep wondering about how [Company Name] is pushing [PERSONALISATION - broad topic, e.g., customer engagement].
    
	We’ve been working with some folks on similar goals, and I’d love to bounce a couple ideas your way—might be a fit. 
    
	Got a minute to talk this week?
""",
"Email 2": """
	Hate to bug you. Is this something I can pass along or no?
""",
"Email 3": """
	Hey [First Name],

	Just wanted to let you know…

	Being an Invite only firm, part of our offer is:

	If we can’t 5x your current exposure in the next 45 days, then we work for FREE until we do.

	Would it make sense to talk about it?
"""
}

BASE_PROMPT = """
    Objective: Generate initial cold emails for outreach, following the specific template provided below. Only modify the sections within square brackets for personalisation; all other content should remain fixed.

    Instructions:

    1. Personalization Fields:
    - Replace [FIRST NAME] with the name of the person given in the prompt.
    - Replace [PERSONALISATION] with a short, specific comment as per the instruction given in square bracket of [PERSONALISATION - instruction here].
    - While personalizing: Write the personalization in 3rd Grade level. The sentence should not be too long and complex. Use shorter sentences and simpler words.
 
    2. Fixed Content:
    - Do not change any other text in the template. All non-bracketed content should remain exactly as written, preserving the wording, tone, and format. Be very very strict on this, I don't want anything else apart from the bracketed  content to change. 

    3. Tone and Language:
    - Keep the tone friendly and professional.
    - Ensure the language is simple, conversational, and concise to stay within a ~150-word limit.

    4. Dont send anything else except for the Email
"""


# Initialize Google Sheets API
creds = service_account.Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=['https://www.googleapis.com/auth/spreadsheets'])
service = build('sheets', 'v4', credentials=creds)

# Initialize OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def summarize_content(content):
    """Summarize content to less than 15 words using OpenAI, or truncate if fails."""
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Summarize the following content to less than 15 words at a 3rd-grade reading level."},
                {"role": "user", "content": content}
            ],
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Summarization failed: {e}")
        words = content.split()
        return ' '.join(words[:14]) + '…' if len(words) > 14 else content

def replace_placeholders(template, data):
    """Replace placeholders in the template with data values."""
    for key, value in data.items():
        template = template.replace(f"[{key}]", value)
    return template

def generate_email(template, data):
    """Generate an email by replacing placeholders in the template."""
    return replace_placeholders(template, data)

def process_row(row):
    """Process a single row and generate emails."""
    # Extract data with defaults
    first_name = row[0]
    company_name = row[3] 
    industry = row[8]
    website = row[6]

    # Collect available data points
    available_data_points = []
    for category in category_ranking:
        col_index = category_indices[category]
        content = row[col_index] if len(row) > col_index else ""
        if content and content not in ["no content", "No URL found", "No meaningful content found"] and len(content.split()) >= 10:
            available_data_points.append(category)

    # Sort by ranking
    available_data_points.sort(key=lambda x: category_ranking[x])

    # Assign to emails (up to 5)
    email_assignments = [None] * 5
    for i in range(min(5, len(available_data_points))):
        email_assignments[i] = available_data_points[i]

    # Generate emails
    emails = {}
    for i, email_type in enumerate(["Email 1", "Subsequence 1", "Subsequence 2", "Subsequence 3", "Subsequence 4"]):
        category = email_assignments[i]
        if category:
            content = row[category_indices[category]].strip()
            personalization = content if len(content.split()) < 15 else summarize_content(content)
            template = prompt_templates[email_type][category] if email_type in ["Email 1", "Subsequence 1"] else prompt_templates[email_type]
        else:
            template = prompt_templates[email_type]["NEUTRAL"] if email_type in ["Email 1", "Subsequence 1"] else prompt_templates[email_type]
            personalization = {
                "Email 1": "your professional expertise",
                "Subsequence 1": "your professional expertise",
                "Subsequence 2": "your great work",
                "Subsequence 3": "",
                "Subsequence 4": "customer engagement"
            }[email_type]

        data = {
            "First Name": first_name,
            "Company Name": company_name,
            "Industry": industry,
            "Personalization": personalization,
            "Website": website,
            "Your Name": "Scale Brands Lab"
        }
        if category == "SERVICES" and email_type in ["Email 1", "Subsequence 1"]:
            data["Industry Problem"] = industry_problems.get(industry, industry_problems["default"])
        if category in ["WEBINAR", "RECENT_BLOG"]:
            data["Specific Topic"] = personalization
        elif category == "PODCAST":
            data["Key Point"] = personalization
        elif category == "EBOOK":
            data["Topic"] = personalization
            data["Specific Section"] = personalization

        emails[email_type] = generate_email(template, data)
        emails[f"{email_type} Data Point"] = category if category else "Neutral"

    # Standard follow-ups
    emails["Email 2"] = prompt_templates["Email 2"]
    emails["Email 3"] = generate_email(prompt_templates["Email 3"], {"First Name": first_name})

    return emails

def main():
    """Main function to process rows and update Google Sheet."""
    # Read data from Sheet2 (A to U)
    range_name = f"{SHEET_NAME}!A2:U"
    result = service.spreadsheets().values().get(spreadsheetId=SPREADSHEET_ID, range=range_name).execute()
    rows = result.get('values', [])

    for i, row in enumerate(rows):
        # Pad row if necessary
        row += [''] * (21 - len(row)) if len(row) < 21 else []
        emails = process_row(row)

        # Prepare values to write
        values = [
            emails["Email 1"],
            emails["Email 1 Data Point"],
            emails["Subsequence 1"],
            emails["Subsequence 1 Data Point"],
            emails["Subsequence 2"],
            emails["Subsequence 2 Data Point"],
            emails["Subsequence 3"],
            emails["Subsequence 3 Data Point"],
            emails["Subsequence 4"],
            emails["Subsequence 4 Data Point"],
            emails["Email 2"],
            emails["Email 3"]
        ]

        # Write to Sheet2 (W to AH)
        write_range = f"{SHEET_NAME}!W{i+2}:AH{i+2}"
        body = {'values': [values]}
        service.spreadsheets().values().update(spreadsheetId=SPREADSHEET_ID, range=write_range, valueInputOption='RAW', body=body).execute()

        print("Columns updated")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'url-to-email-445616-cebe4868914f.json'